# LexAI / Legal RAG Pipeline Overview

This notebook documents the end‑to‑end pipeline for building the legal search system.
It mirrors the Python scripts in the `pipeline/` package:

- `scraper.py`
- `preprocessing.py`
- `chunking.py`
- `db_init.py`
- `vectorize.py`
- `upsert.py`
- `similarity_search.py`
- `reranking.py`

## 1. Project Setup

Configure paths and environment variables used by the pipeline. Adjust this to match your actual project structure (e.g. `PYTHONPATH`, `.env` location, DB URL, etc.).

In [1]:
import os
from pathlib import Path

# Project root (assumes this notebook lives at repo root or in a `notebooks/` folder)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PIPELINE_ROOT = PROJECT_ROOT / 'pipeline'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PIPELINE_ROOT:', PIPELINE_ROOT)

# Optional: point to .env for your backend / pipeline
ENV_PATH = PROJECT_ROOT / '.env'
if ENV_PATH.exists():
    print('Loading environment from', ENV_PATH)
    from dotenv import load_dotenv
    load_dotenv(ENV_PATH)
else:
    print('No .env file found at', ENV_PATH)

# Optional: ensure pipeline package is importable
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print('Added to sys.path:', PROJECT_ROOT)


PROJECT_ROOT: /root/dev/PhilLexRAG
PIPELINE_ROOT: /root/dev/PhilLexRAG/pipeline
Loading environment from /root/dev/PhilLexRAG/.env
Added to sys.path: /root/dev/PhilLexRAG


## 2. High‑Level Architecture

At a high level, the legal RAG pipeline looks like this:

1. **Scraping (`scraper.py`)** – Download raw decisions HTML from the Supreme Court e‑Library.
2. **Preprocessing (`preprocessing.py`)** – Clean text, normalize whitespace, remove boilerplate, extract metadata.
3. **Chunking (`chunking.py`)** – Split long documents into overlapping chunks suitable for embeddings.
4. **Database Initialization (`db_init.py`)** – Create relational tables + `pgvector` extension (or SQLite equivalent).
5. **Vectorization (`vectorize.py`)** – Compute embeddings for each chunk `BGE‑M3`.
6. **Upsert (`upsert.py`)** – Write chunks + embeddings into the database.
7. **Similarity Search (`similarity_search.py`)** – Query‑time vector search over chunks.
8. **Re‑ranking (`reranking.py`)** – Second‑stage ranking (e.g. cross‑encoder, LLM score) on top‑k hits.

The next sections show how each step can be invoked from the notebook.

## 3. Scraping (pipeline/scraper.py)

This step crawls the target sources (e‑Library, etc.) and stores raw documents locally, storing it in a `data` folder.  

In [ ]:
output_file_raw = PROJECT_ROOT / 'data' / 'sc_decisions.jsonl'
cache_dir = PROJECT_ROOT / 'data' / 'cache'
checkpoint_file = PROJECT_ROOT / 'data' / 'checkpoint_done.txt'

if not Path(output_file_raw).exists():
    from pipeline import scraper

    scraper.crawl_decisions(output_file_raw, cache_dir, checkpoint_file)
else:
    print("Already done scraping!")

Already done scraping!


## 4. Preprocessing (pipeline/preprocessing.py)

Preprocessing turns raw text into cleaned, structured documents. Typical operations:

- Strip headers/footers and boilerplate
- Save outputs to `data/`


In [13]:
import json
from pipeline import preprocessing
output_file_cleaned = PROJECT_ROOT / 'data' / 'sc_decisions_cleaned.jsonl'

if not Path(output_file_cleaned).exists():
    with open(output_file_raw, "r", encoding="utf-8") as fin, \
        open(output_file_cleaned, "w", encoding="utf-8") as fout:
        for line in fin:
            obj = json.loads(line)

            if "text" in obj:
                obj["text"] = preprocessing.cut_before_division(obj["text"])

                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
else:
    print("Already done preprocessing!")

Already done preprocessing!


## 5. Database Initialization (pipeline/db_init.py)

This step sets up the relational schema and vector index:

- Enable `pgvector` extension for Postgres
- Create `documents`, `chunks`, and `chunk_embeddings` tables
- Define IVFFlat index


In [ ]:
from pipeline import db_init

conn = db_init.create_connection()
db_init.run_db_init(conn)
db_init.close_connection(conn)

## 6. Chunking (pipeline/chunking.py) and Upsert (pipeline/upsert.py)

Chunking splits cleaned documents into overlapping segments:

- `max_tokens` set as 350
- `overlap_sentences` set as 2

The output is usually a table or Parquet/JSONL file with one row per chunk containing:

- `case_no`
- `division`
- `title`
- `section`
- `chunk_index`
- `text`

In [17]:
from pipeline import chunking
from pipeline import upsert

chunking_checkpoint = Path(PROJECT_ROOT / 'data' / 'chunking_checkpoint.txt')

last_url = chunking_checkpoint.read_text().strip() if chunking_checkpoint.exists() else None
resume_mode = last_url is not None

print(f"▶ Resume mode: {resume_mode} | last URL: {last_url}")

skip = resume_mode
if not chunking_checkpoint.exists():
    with output_file_cleaned.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f):
            checkpoint = True
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"⚠ JSON error at line {line_number}: {e}")
                continue

            year = str(rec.get("year", ""))
            month = str(rec.get("month", ""))
            title = rec["title"]
            url = rec["url"]
            text = rec["text"]

            # Skip until we see the last processed URL
            if skip:
                if url == last_url:
                    skip = False  # stop skipping AFTER this record
                continue

            print(f"🔹 Processing record {line_number} → {url}")

            chunks = chunking.build_rag_chunks(
                text,
                max_tokens=350,
                overlap_sentences=2,
            )

            for ch in chunks:
                try:
                    cid = upsert.insert_chunk_safe(conn, ch)
                    print("   ✅ Inserted chunk id:", cid)
                except Exception as e:
                    print("   ❌ Failed to insert chunk:", e)
                    checkpoint = False
                    break # stop processing further chunks for this record
            
            if checkpoint:
                # Save checkpoint as last processed unique URL
                chunking_checkpoint.write_text(url)

        print("✅ Chunking complete!")

else:
    print("Already completed chunking!")

▶ Resume mode: True | last URL: https://elibrary.judiciary.gov.ph/thebookshelf/showdocs/1/48857
Already completed chunking!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 7. Vectorization and Upsert (pipeline/vectorize.py & pipeline/upsert.py)

Vectorization converts each chunk into an embedding using BGE‑M3.  
Upsert then writes the chunk + embedding into the database.

Common design:

- `vectorize.py` – Pure embedding logic (`embed_texts` function, model client, batching, rate limiting).
- `upsert.py` – Reads chunk table/file, calls `embed_texts`, and inserts into DB.


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from pipeline import vectorize

conn = db_init.create_connection()
model = SentenceTransformer("BAAI/bge-m3")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

with conn.cursor() as cur:
    cur.execute("SELECT id, text FROM decision_chunks WHERE embedding IS NULL;")
    rows = cur.fetchall()

print(f"▶ Found {len(rows)} chunks to embed.")

for row_id, txt in rows:
    print(f"Embedding chunk id {row_id}")
    try:
        emb = vectorize.encode_passage(txt, model)
        tokens = vectorize.count_tokens(txt, tokenizer)

        with conn.cursor() as cur:
            cur.execute(
                """
                UPDATE decision_chunks
                SET embedding = %s::vector,
                    token_count = %s
                WHERE id = %s;
                """,
                (emb.tolist(), tokens, row_id),
            )
    except Exception as e:
        print(f"❌ Failed to embed chunk id {row_id}: {e}")

## 8. Similarity Search Demo (pipeline/similarity_search.py)

Once the DB is populated, similarity search retrieves the most relevant chunks for a given query.  
This section is useful for the write‑up because it shows an end‑to‑end retrieval example.

In [ ]:
from sentence_transformers import SentenceTransformer
from pipeline import similarity_search, db_init
from pgvector.psycopg import register_vector 

conn = db_init.create_connection()
register_vector(conn)
model = SentenceTransformer("BAAI/bge-m3").to_cuda()

text = "child abuse"

hits = similarity_search.search_chunks(conn, model, text, 5)
for h in hits:
    print(h["case_no"], h["section"], h["chunk_index"], "similarity:", h["similarity"], h["text"])
db_init.close_connection(conn)

✅ Connection successful!


## 9. Future Work: Re‑ranking (pipeline/reranking.py)

Re‑ranking is a planned second‑stage model that refines the top‑k hits from similarity search.  
Possible directions:

- Cross‑encoder (e.g. `bge‑reranker`, `ms‑marco‑miniLM`) that scores **(query, chunk)** pairs.
- LLM‑based scoring (ask the LLM to rate each chunk's usefulness for the query).
- Hybrid: first BM25 + vectors, then re‑rank with a cross‑encoder.

A typical interface for `reranking.py` might look like:

```python
def rerank(query: str, candidates: list[RetrievedChunk], top_k_rerank: int = 10) -> list[RetrievedChunk]:
    ...
```

Once implemented, you can add a short demo cell here that:

1. Calls `similarity_search.search(query, top_k=50)`.
2. Calls `reranking.rerank(query, candidates, top_k_rerank=10)`.
3. Prints the final top‑10 with scores.
